# WaterPulse — ETL & Data Cleaning Notebook
**Dataset:** MIS 78th Round — Improved Drinking Water Source Indicators (AI Kosh)
**Platform:** IBM Watson Studio (Lite Plan)
**Author:** WaterPulse Data Engineering Team

## Pipeline
1. Load raw CSV from IBM Cloud Object Storage
2. Standardize state names, handle missing values
3. Compute derived equity metrics
4. Load cleaned tables into IBM Db2 on Cloud (Lite)


In [ ]:
# ── Install dependencies (Watson Studio Lite environment) ─────────────────────
import subprocess
subprocess.run(['pip', 'install', 'ibm-cos-sdk', 'ibm-db', 'ibm-watson', 'pandas', 'numpy', 'openpyxl'], check=True)

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import ibm_boto3
from ibm_botocore.client import Config
import ibm_db
import io
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully')

In [ ]:
# ── IBM Cloud Object Storage Configuration (Lite: 25 GB free) ────────────────
# Replace with your actual COS credentials from IBM Cloud > Manage > Access (IAM)
COS_API_KEY      = 'YOUR_COS_API_KEY'
COS_INSTANCE_CRN = 'YOUR_COS_INSTANCE_CRN'
COS_ENDPOINT     = 'https://s3.us-south.cloud-object-storage.appdomain.cloud'
COS_BUCKET       = 'waterpulse-raw-data'

cos_client = ibm_boto3.client(
    service_name='s3',
    ibm_api_key_id=COS_API_KEY,
    ibm_service_instance_id=COS_INSTANCE_CRN,
    config=Config(signature_version='oauth'),
    endpoint_url=COS_ENDPOINT
)

def load_from_cos(bucket: str, key: str) -> pd.DataFrame:
    """Download an object from COS and return as DataFrame."""
    obj = cos_client.get_object(Bucket=bucket, Key=key)
    if key.endswith('.xlsx'):
        return pd.read_excel(io.BytesIO(obj['Body'].read()))
    return pd.read_csv(io.BytesIO(obj['Body'].read()))

# Load MIS 78th Round sheets
df_water   = load_from_cos(COS_BUCKET, 'mis78_drinking_water.csv')
df_cooking = load_from_cos(COS_BUCKET, 'mis78_clean_cooking_fuel.csv')
df_migr    = load_from_cos(COS_BUCKET, 'mis78_migration.csv')

print(f'Water rows: {len(df_water)} | Cooking rows: {len(df_cooking)} | Migration rows: {len(df_migr)}')

In [ ]:
# ── State Name Standardization Map ───────────────────────────────────────────
# MIS 78th Round uses Census 2011 state codes; normalize to consistent names
STATE_MAP = {
    'A&N Islands': 'Andaman & Nicobar Islands',
    'A & N Islands': 'Andaman & Nicobar Islands',
    'D&N Haveli': 'Dadra & Nagar Haveli and Daman & Diu',
    'D & N Haveli': 'Dadra & Nagar Haveli and Daman & Diu',
    'Daman & Diu': 'Dadra & Nagar Haveli and Daman & Diu',
    'Jammu & Kashmir': 'Jammu & Kashmir',
    'J&K': 'Jammu & Kashmir',
    'NCT Delhi': 'Delhi',
    'Delhi': 'Delhi',
    'Orissa': 'Odisha',
    'Uttaranchal': 'Uttarakhand',
    'Chhattisgarh': 'Chhattisgarh',
}

STATE_CODE_MAP = {
    1: 'Jammu & Kashmir', 2: 'Himachal Pradesh', 3: 'Punjab', 4: 'Chandigarh',
    5: 'Uttarakhand', 6: 'Haryana', 7: 'Delhi', 8: 'Rajasthan', 9: 'Uttar Pradesh',
    10: 'Bihar', 11: 'Sikkim', 12: 'Arunachal Pradesh', 13: 'Nagaland', 14: 'Manipur',
    15: 'Mizoram', 16: 'Tripura', 17: 'Meghalaya', 18: 'Assam', 19: 'West Bengal',
    20: 'Jharkhand', 21: 'Odisha', 22: 'Chhattisgarh', 23: 'Madhya Pradesh',
    24: 'Gujarat', 25: 'Dadra & Nagar Haveli and Daman & Diu', 26: 'Maharashtra',
    27: 'Andhra Pradesh', 28: 'Karnataka', 29: 'Goa', 30: 'Lakshadweep',
    31: 'Kerala', 32: 'Tamil Nadu', 33: 'Puducherry', 34: 'Andaman & Nicobar Islands',
    35: 'Telangana', 36: 'Ladakh'
}

def normalize_state(name: str) -> str:
    if pd.isna(name): return 'Unknown'
    name = str(name).strip()
    return STATE_MAP.get(name, name)

for df in [df_water, df_cooking, df_migr]:
    if 'state_name' in df.columns:
        df['state_name'] = df['state_name'].apply(normalize_state)
    if 'state_code' in df.columns:
        df['state_code'] = pd.to_numeric(df['state_code'], errors='coerce').astype('Int64')

print('State names normalized')

In [ ]:
# ── MIS 78th Round — Water Indicator Cleaning ─────────────────────────────────
# Expected columns from AI Kosh export:
#   state_code, state_name, sector (Rural/Urban/Total),
#   social_group (ST/SC/OBC/Others/Total),
#   pct_improved_water_source, pct_piped_supply, pct_treated_water,
#   sample_hh, year

WATER_COLS = [
    'state_code', 'state_name', 'sector', 'social_group',
    'pct_improved_water_source', 'pct_piped_supply',
    'pct_treated_water', 'sample_hh', 'year'
]

# Rename columns to standard schema if needed
col_renames = {
    'State Code': 'state_code', 'State': 'state_name', 'Sector': 'sector',
    'Social Group': 'social_group',
    'Improved Source of Drinking Water (%)': 'pct_improved_water_source',
    'Piped Water Supply (%)': 'pct_piped_supply',
    'Treated Water (%)': 'pct_treated_water',
    'Sample Households': 'sample_hh', 'Year': 'year'
}
df_water.rename(columns=col_renames, inplace=True)

# Fill missing pct values with median per state-sector group
for col in ['pct_improved_water_source', 'pct_piped_supply', 'pct_treated_water']:
    if col in df_water.columns:
        df_water[col] = pd.to_numeric(df_water[col], errors='coerce')
        df_water[col] = df_water.groupby(['state_code', 'sector'])[col].transform(
            lambda x: x.fillna(x.median())
        )

# Drop rows still missing critical values
df_water_clean = df_water.dropna(subset=['state_code', 'pct_improved_water_source'])

# Standardize sector column
df_water_clean['sector'] = df_water_clean['sector'].str.strip().str.title()

print(f'Cleaned water data: {len(df_water_clean)} rows')
df_water_clean.head()

In [ ]:
# ── Derived Equity Metrics ────────────────────────────────────────────────────

# Pivot to get urban and rural side-by-side for equity gap computation
df_total = df_water_clean[df_water_clean['social_group'].str.upper() == 'TOTAL']

pivot = df_total.pivot_table(
    index=['state_code', 'state_name', 'year'],
    columns='sector',
    values='pct_improved_water_source'
).reset_index()

pivot.columns.name = None
pivot = pivot.rename(columns={'Rural': 'rural_pct', 'Urban': 'urban_pct', 'Total': 'total_pct'})

# Equity gap: positive = urban better off (typical scenario)
pivot['equity_gap']  = pivot['urban_pct'] - pivot['rural_pct']

# SDG 6.1 proxy score: % population with at least basic water service
# Weight: Rural population is ~65% of India's population
pivot['sdg61_proxy'] = (pivot['rural_pct'] * 0.65) + (pivot['urban_pct'] * 0.35)

# YoY change (requires multi-year data; compute if available)
pivot = pivot.sort_values(['state_code', 'year'])
pivot['yoy_change'] = pivot.groupby('state_code')['total_pct'].diff()

# SDG risk flag: states where sdg61_proxy < 80 OR equity_gap > 15
pivot['sdg_risk_flag'] = (
    (pivot['sdg61_proxy'] < 80) | (pivot['equity_gap'] > 15)
).astype(int)

print('Derived metrics computed:')
print(pivot[['state_name', 'rural_pct', 'urban_pct', 'equity_gap', 'sdg61_proxy', 'sdg_risk_flag']].head(10))

In [ ]:
# ── Clean Cooking Fuel Data ───────────────────────────────────────────────────
cook_renames = {
    'State Code': 'state_code', 'State': 'state_name', 'Sector': 'sector',
    'Social Group': 'social_group',
    'Households using Clean Fuel (%)': 'pct_clean_cooking',
    'Households using LPG/PNG (%)': 'pct_lpg_png',
    'Year': 'year'
}
df_cooking.rename(columns=cook_renames, inplace=True)
df_cooking['pct_clean_cooking'] = pd.to_numeric(df_cooking.get('pct_clean_cooking', np.nan), errors='coerce')

# Merge with water data on state + year + sector
df_merged = pivot.merge(
    df_cooking[df_cooking['social_group'].str.upper() == 'TOTAL']
        .pivot_table(index=['state_code','year'], columns='sector',
                     values='pct_clean_cooking').reset_index()
        .rename(columns={'Rural':'cook_rural_pct','Urban':'cook_urban_pct'}),
    on=['state_code', 'year'], how='left'
)

print(f'Merged water + cooking fuel: {len(df_merged)} rows')

In [ ]:
# ── Migration Data Integration ────────────────────────────────────────────────
migr_renames = {
    'State Code': 'state_code', 'State': 'state_name',
    'Migration Rate (%)': 'migration_rate',
    'Net Migration (000s)': 'net_migration_000s',
    'Year': 'year'
}
df_migr.rename(columns=migr_renames, inplace=True)
df_migr['migration_rate'] = pd.to_numeric(df_migr.get('migration_rate', np.nan), errors='coerce')

df_final = df_merged.merge(
    df_migr[['state_code','year','migration_rate','net_migration_000s']],
    on=['state_code','year'], how='left'
)

print(f'Final dataset shape: {df_final.shape}')
print(df_final.dtypes)
df_final.head()

In [ ]:
# ── Save cleaned data back to COS ─────────────────────────────────────────────
csv_buffer = io.BytesIO()
df_final.to_csv(csv_buffer, index=False)
csv_buffer.seek(0)

cos_client.put_object(
    Bucket=COS_BUCKET,
    Key='cleaned/waterpulse_master_cleaned.csv',
    Body=csv_buffer.getvalue()
)
print('Cleaned dataset saved to COS: cleaned/waterpulse_master_cleaned.csv')

In [ ]:
# ── Load to IBM Db2 on Cloud (Lite: 200 MB, 5 tables) ────────────────────────
DB2_DSN = (
    'DATABASE=BLUDB;'
    'HOSTNAME=YOUR_DB2_HOST.databases.appdomain.cloud;'
    'PORT=30376;'
    'PROTOCOL=TCPIP;'
    'UID=YOUR_USERNAME;'
    'PWD=YOUR_PASSWORD;'
    'SECURITY=SSL;'
)

conn = ibm_db.connect(DB2_DSN, '', '')

# Create table DDL
DDL = """
CREATE TABLE IF NOT EXISTS WATERPULSE.STATE_WATER_METRICS (
    state_code       SMALLINT,
    state_name       VARCHAR(80),
    year             SMALLINT,
    rural_pct        DECIMAL(5,2),
    urban_pct        DECIMAL(5,2),
    total_pct        DECIMAL(5,2),
    equity_gap       DECIMAL(5,2),
    sdg61_proxy      DECIMAL(5,2),
    yoy_change       DECIMAL(5,2),
    sdg_risk_flag    SMALLINT,
    cook_rural_pct   DECIMAL(5,2),
    cook_urban_pct   DECIMAL(5,2),
    migration_rate   DECIMAL(5,2),
    net_migration_000s DECIMAL(8,1),
    PRIMARY KEY (state_code, year)
)
"""

try:
    ibm_db.exec_immediate(conn, DDL)
    print('Table WATERPULSE.STATE_WATER_METRICS created or already exists')
except Exception as e:
    print(f'DDL note: {e}')

In [ ]:
# ── Batch insert into Db2 ─────────────────────────────────────────────────────
INSERT_SQL = """
INSERT INTO WATERPULSE.STATE_WATER_METRICS
    (state_code, state_name, year, rural_pct, urban_pct, total_pct,
     equity_gap, sdg61_proxy, yoy_change, sdg_risk_flag,
     cook_rural_pct, cook_urban_pct, migration_rate, net_migration_000s)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
"""

stmt = ibm_db.prepare(conn, INSERT_SQL)
rows_inserted = 0

for _, row in df_final.iterrows():
    try:
        ibm_db.execute(stmt, (
            int(row.get('state_code', 0)),
            str(row.get('state_name', '')),
            int(row.get('year', 2023)),
            float(row.get('rural_pct', 0) or 0),
            float(row.get('urban_pct', 0) or 0),
            float(row.get('total_pct', 0) or 0),
            float(row.get('equity_gap', 0) or 0),
            float(row.get('sdg61_proxy', 0) or 0),
            float(row.get('yoy_change', 0) or 0),
            int(row.get('sdg_risk_flag', 0) or 0),
            float(row.get('cook_rural_pct', 0) or 0),
            float(row.get('cook_urban_pct', 0) or 0),
            float(row.get('migration_rate', 0) or 0),
            float(row.get('net_migration_000s', 0) or 0)
        ))
        rows_inserted += 1
    except Exception as e:
        print(f'Row insert error: {e}')

ibm_db.close(conn)
print(f'Inserted {rows_inserted} rows into Db2 STATE_WATER_METRICS')